# Molecular Property Prediction with Chemprop
## AI for Science Demo — Drug Discovery Workshop

**Core Question:** Given a molecule's SMILES structure, can we predict ADMET-related molecular properties using a graph neural network?

**Approach:** Chemprop (MPNN) trained on two MoleculeNet datasets:

- **ESOL** → predict LogS (aqueous solubility)
- **Lipophilicity** → predict logD (lipophilicity)

## 1. Setup & Imports

In [5]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from IPython.display import Image, display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from scripts.molecule_cases import DATASETS, load_model, predict as predict_with_model, property_label

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  RDKit {Chem.rdkitVersion}")

ModuleNotFoundError: No module named 'rdkit'

## 2. Load Trained Models + Scalers

The scaler is **critical**: training targets were normalized (mean=0, std=1).
We rebuild the scaler from each training split so predictions can be transformed back to the original target scale:

- ESOL: LogS
- Lipophilicity: logD

In [3]:
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

RESULT_FILES = {
    "esol": os.path.join(MODELS_DIR, "results_esol_train.json"),
    "lipo": os.path.join(MODELS_DIR, "results_lipo_train.json"),
}

loaded = {}
metrics = {}

for dataset_key, config in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"Loading {config['display_name']} model")
    print(f"{'='*60}")

    model, device, scaler = load_model(dataset_key, config)
    if model is None:
        print(f"SKIP {config['display_name']}: run `python scripts/train.py` first.")
        continue

    loaded[dataset_key] = {
        "model": model,
        "device": device,
        "scaler": scaler,
        "config": config,
    }

    result_path = RESULT_FILES[dataset_key]
    if os.path.exists(result_path):
        with open(result_path) as f:
            result = json.load(f)
        metrics[dataset_key] = result
        print(f"\nModel Performance on Test Set:")
        print(f"  RMSE = {result['rmse']:.3f} {config['target_label']}")
        print(f"  MAE  = {result['mae']:.3f} {config['target_label']}")
        print(f"  R²   = {result['r2']:.3f}")
        print(f"  Trained on {result.get('n_train', '?')} molecules")
    else:
        print(f"WARNING: results file not found: {result_path}")

print(f"\nReady for inference. Loaded datasets: {', '.join(loaded.keys())}")

NameError: name 'PROJECT_ROOT' is not defined

## 3. Inference Function

Set `dataset="esol"` to predict LogS, or `dataset="lipo"` to predict logD.

In [ ]:
def explain_prediction(dataset_key: str, value: float) -> str:
    """Return a short interpretation for the predicted property value."""
    if dataset_key == "esol":
        if value > 0:
            return "Dissolves readily. Good for oral drug formulation."
        if value > -2:
            return "Acceptable for many drug development purposes."
        if value > -4:
            return "May need formulation strategies such as salt forms or nanoparticles."
        return "Major challenge for oral bioavailability."

    if value < 1:
        return "More hydrophilic; may have weaker membrane permeability."
    if value < 3:
        return "Balanced lipophilicity; often a useful range for drug-like molecules."
    if value < 4:
        return "Lipophilic; watch for solubility and nonspecific binding issues."
    return "Very lipophilic; likely solubility and ADMET risk."


def predict_property(smiles: str, dataset: str = "esol", show_structure: bool = True):
    """Predict LogS (ESOL) or logD (Lipophilicity) for a given SMILES string."""
    if dataset not in loaded:
        raise ValueError(f"Dataset '{dataset}' is not loaded. Available: {list(loaded.keys())}")

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    bundle = loaded[dataset]
    config = bundle["config"]
    value = predict_with_model(bundle["model"], bundle["device"], bundle["scaler"], smiles)
    level = property_label(dataset, value)

    if show_structure:
        display(Draw.MolToImage(mol, size=(300, 200)))

    print(f"Dataset:          {config['display_name']}")
    print(f"SMILES:           {smiles}")
    print(f"Predicted {config['target_label']}: {value:.2f}")
    print(f"Interpretation:   {level}")
    print(f"  -> {explain_prediction(dataset, value)}")

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    print(f"\nMolecular Properties (RDKit):")
    print(f"  MW: {mw:.1f} g/mol  |  RDKit LogP: {logp:.2f}  |  HBD: {hbd}  |  HBA: {hba}")

    return value


def predict_logS(smiles: str, show_structure: bool = True):
    """Backward-compatible helper for ESOL solubility prediction."""
    return predict_property(smiles, dataset="esol", show_structure=show_structure)

## 4. Live Demo — Predict One Molecule

Enter any SMILES string below and choose a dataset:

- `dataset_key = "esol"` predicts aqueous solubility / LogS
- `dataset_key = "lipo"` predicts lipophilicity / logD

In [ ]:
# ── DEMO: Try your own molecule! ──
# Replace the SMILES below with any valid SMILES string.
# Change dataset_key to "esol" or "lipo".

my_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin
dataset_key = "esol"

predicted_value = predict_property(my_smiles, dataset=dataset_key)

### Try These Well-Known Molecules

In [ ]:
demo_molecules = {
    "Aspirin":      "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":     "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Benzene":      "c1ccccc1",
    "Ibuprofen":    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Paracetamol":  "CC(=O)NC1=CC=C(C=C1)O",
    "Glucose":      "C(C1C(C(C(C(O1)O)O)O)O)O",
    "Testosterone": "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",
    "Vitamin C":    "C(C(C1C(=C(C(=O)O1)O)O)O)O",
}

predictions = []
for name, smi in demo_molecules.items():
    row = {"Molecule": name}
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    for dataset_key in loaded.keys():
        try:
            value = predict_property(smi, dataset=dataset_key, show_structure=False)
            label = loaded[dataset_key]["config"]["target_label"]
            row[f"Predicted {label}"] = f"{value:.2f}"
        except Exception as e:
            row[dataset_key] = f"Error: {e}"
    predictions.append(row)

if predictions:
    print(f"\n{'='*60}")
    print("  Summary")
    print(f"{'='*60}")
    df = pd.DataFrame(predictions)
    display(df)

## 5. Results Overview (Pre-generated Figures)

Run `python scripts/molecule_cases.py` and `python scripts/visualize.py` first to generate all figures.

In [ ]:
FIGURES = [
    ("Predicted vs True", "predicted_vs_true.png", 500),
    ("Error Distribution", "error_distribution.png", 550),
    ("Effect of Training Data Size", "data_scale_rmse.png", 700),
    ("Training Curves", "training_curves.png", 500),
    ("Molecule Examples", "molecule_examples.png", 700),
]

for dataset_key, config in DATASETS.items():
    prefix = "" if dataset_key == "esol" else f"{dataset_key}_"
    print(f"\n{'='*60}")
    print(config["display_name"])
    print(f"{'='*60}")

    for title, filename, width in FIGURES:
        path = os.path.join(RESULTS_DIR, f"{prefix}{filename}")
        print(f"\n{title}")
        if os.path.exists(path):
            display(Image(path, width=width))
        else:
            print(f"  Missing: {path}")
            print("  Run README Step 4 and Step 5 first.")

## 6. Key Takeaways

1. **Graph Neural Networks** treat molecules as graphs (atoms = nodes, bonds = edges), naturally capturing molecular topology without hand-crafted features.

2. **The same MPNN pipeline generalizes across molecular properties:** ESOL predicts aqueous solubility (LogS), while Lipophilicity predicts logD.

3. **Data quantity matters:** The data-scale experiments test how performance changes from 20% to 100% of the training data for both datasets.

4. **ADMET properties are coupled:** Solubility and lipophilicity form a useful contrast for drug discovery: one reflects water compatibility, the other membrane/partition behavior.

5. **Chemprop lowers the barrier:** Modular, PyTorch-based tooling makes it practical to run molecular property prediction experiments on a personal computer.

---
*References: Delaney, J.S. J. Chem. Inf. Comput. Sci. 2004, 44, 1000–1005 | MoleculeNet Lipophilicity | Chemprop v2: Heid et al. J. Chem. Inf. Model. 2025*